In [192]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [76]:
class VanillaAttention(nn.Module):
    def __init__(self, emb=2):
        super().__init__()

        self.emb = emb

        self.W_k = nn.Linear(emb, emb, bias=False)
        self.W_q = nn.Linear(emb, emb, bias=False)
        self.W_v = nn.Linear(emb, emb, bias=False)

    def forward(self, X):

        b, t, e = X.size()
        
        assert e == self.emb, f'Input embedding dim ({e}) should match layer embedding dim ({self.emb})'
        
        # query, key, value model
        K = self.W_k(X)
        Q = self.W_q(X)
        V = self.W_v(X)

        W = Q@K.transpose(1,2)
        W = F.softmax(W, dim=2)

        Y = W@V
        return Y
    
class LightTransformer(nn.Module):

    def __init__(self, emb=2, ff_hidden_mult=32):
        super().__init__()
        self.attention = VanillaAttention(emb)
        self.ff = nn.Sequential(
            nn.Linear(emb, ff_hidden_mult * emb),
            nn.ReLU(),
            nn.Linear(ff_hidden_mult * emb, emb))
    def forward(self, x):

        attended = self.attention(x)
        fedforward = self.ff(x)
        return x
    
class Svetty(nn.Module):
    def __init__(self,N_BLOCKS,
                EMB_DIM, N_OUT,
                 HIDDEN_MULT, DEVICE):
        super().__init__()
        self.emb_dim = EMB_DIM
        self.n_out = N_OUT
        self.transformer_blocks = [LightTransformer(EMB_DIM,
                                                  HIDDEN_MULT) for _ in range(N_BLOCKS)]
        self.transformer_blocks = nn.Sequential(*self.transformer_blocks)
        self.head =  nn.Linear(EMB_DIM, N_OUT).to(DEVICE)
        
    def total_params(self):
        pp=0
        for p in list(self.parameters()):
            nn=1
            for s in list(p.size()):
                nn = nn*s
            pp += nn
        return pp
    
    def forward(self, X):
        B, T, E = X.shape
        X = self.transformer_blocks(X)
        X = X.mean(1)
        Y = self.head(X)
        return Y

In [177]:
model = Svetty(10, 2, 1, 32, "cpu")

In [178]:
model.total_params()

3343